
#### Requisitos para o arquivo torneios.csv

Objetivo: realizar um processo de ETL e disponibilizar os dados em uma camada apropriada

Ler -> Transformar -> Escrever (ETL)

1. Ler o arquivo csv do Data Lake
2. Renomear as colunas para português utilizando o **snake case**
3. Definir o esquema de dados correto
4. Precisa ter uma coluna com a data em que o arquivo foi consumido
5. Salvar os dados em formato **parquet** na camada adequada
6. Bônus: Criar uma coluna com a palavra 'Brasil'


#### 1. Lendo o arquivo CSV

In [0]:
dbutils.widgets.text("arquivo_fonte", "")
meu_arquivo_fonte = dbutils.widgets.get("arquivo_fonte")

In [0]:
print(meu_arquivo_fonte)

In [0]:
%run "../Modulo 2/Configurando o storage"

In [0]:
%run "../Modulo 4/Variaveis"

In [0]:
%run "../Modulo 4/funcoes"

In [0]:
caminho_arquivo_torneios = f"{caminho_bronze}{meu_arquivo_fonte}.csv"


In [0]:
display(dbutils.fs.ls(caminho_bronze))

In [0]:
df = (spark.read
               .option("encoding", "UTF-16")
               .option("header", "true")
               .option("sep", ",")
               .option("inferSchema", "true")
               .csv(caminho_arquivo_torneios))

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
if meu_arquivo_fonte == "torneios":
    df_renomeado = df \
        .withColumnRenamed("league_id", "id_torneio") \
        .withColumnRenamed("league_name", "nm_torneio") \
        .withColumnRenamed("country_name", "nm_pais") \
        .withColumnRenamed("country_code", "cd_pais") \
        .withColumnRenamed("season_year", "ano_torneio") \
        .withColumnRenamed("season_start", "dt_inicio") \
        .withColumnRenamed("season_end", "dt_fim") \
        .withColumnRenamed("season_current", "e_atual")
elif meu_arquivo_fonte == "estadios":
    df_renomeado = df \
        .withColumnRenamed("id", "id_estadio") \
        .withColumnRenamed("name", "nm_estadio") \
        .withColumnRenamed("address", "endereco_estadio") \
        .withColumnRenamed("city", "nm_cidade") \
        .withColumnRenamed("country", "nm_pais") \
        .withColumnRenamed("capacity", "vl_capacidade") \
        .withColumnRenamed("surface", "tp_grama") \
        .withColumnRenamed("image", "link_imagem")
    

In [0]:
df_renomeado.display()

In [0]:
from pyspark.sql.types import BooleanType

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
df_data = cria_coluna_data(df_renomeado)

In [0]:
display(df_data)

In [0]:
df_data.write.parquet(f"{caminho_silver}{meu_arquivo_fonte  }/", mode="overwrite")

In [0]:
dbutils.notebook.exit(f"O arquivo {meu_arquivo_fonte} foi processado com sucesso")